## Storage: volumes, PVs & PVCs

A container's writable layer is **deleted when the container is removed** — restart your database pod and the data is gone. That's fine for cattle, fatal for state. This module gives pods storage that *outlives* them.

Kubernetes offers two levels:

- **Pod-lifetime storage** — a **volume** that lives as long as the pod: `emptyDir`, a ConfigMap/Secret mount. Scratch space, sidecar IPC, mounted config.
- **Cluster-lifetime storage** — backed by something *external* (a cloud disk, NFS) that survives pod deletion, node failure, and rescheduling. The **PersistentVolume / PersistentVolumeClaim** abstraction.

The central design is a **split of concerns**:

```
Pod  →references→  PVC (namespaced: "I want 5Gi RWO")  →bound to→  PV (cluster: "here's the disk")
                        ↑ StorageClass provisions PVs on demand (CSI driver)
```

The app says *what* it needs; the PV says *what's available*; a controller binds them; a **CSI driver** does the real work. The app never knows if it got an EBS volume or an NFS share.

We'll cover volumes, ephemeral types, the PV/PVC/StorageClass triangle, access modes, reclaim policies, the CSI, and how **StatefulSets** give each pod its own PVC. On our map this whole module is the **Storage** box in the Resources band — **PV**, **PVC**, **StorageClass**, **CSI** — resolving onto the Pods that mount it.